# Taxi Trip Data Cleaning

Steps:
1. Load raw CSV
2. Type conversions & trip end filling
3. Rename to snake_case
4. Impute lat/lon from census tract
5. Impute census tract from lat/lon (lookup)
6. Drop rows with no spatial data
7. Payment type cleanup
8. Company standardization
9. Duplicate check
10. Remove invalid trips (any zero/below-threshold)
11. Consistency & logic checks
12. Absolute outlier filtering
13. Save cleaned parquet
14. ID remapping + compact parquet
15. Time features from trip_start
16. Cyclic encoding (sin/cos)
17. Electronic payment fee detection + save compact parquet
18. Save removed rows

To run this notebook it is necessary to have the Taxi_Trip data downloaded as csv and names as "<i>Taxi_Trips_(2024-).csv.</i>"

## 0) Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import os
from pathlib import Path

PROJECT = Path(os.getcwd()).parent
DATA_DIR = PROJECT / "data"

CSV_PATH = DATA_DIR / "Taxi_Trips_(2024-).csv"
OUTPUT_CLEAN = DATA_DIR / "Taxi_Trips_cleaned.parquet"
OUTPUT_COMPACT = DATA_DIR / "Taxi_Trips_compact.parquet"
OUTPUT_TRIP_ID_MAP = DATA_DIR / "trip_id_mapping.csv"
OUTPUT_TAXI_ID_MAP = DATA_DIR / "taxi_id_mapping.csv"
OUTPUT_REMOVED = DATA_DIR / "Removed_Rows.parquet"
OUTPUT_NO_SPATIAL = DATA_DIR / "No_Spatial_Value.parquet"
OUTPUT_COMPANIES = DATA_DIR / "company_names_after_cleaning.csv"

removed_frames = []

def flag_and_remove(df, mask, reason):
    n = mask.sum()
    if n == 0:
        return df
    removed = df.loc[mask].copy()
    removed["removal_reason"] = reason
    removed_frames.append(removed)
    print(f"  Removed {n:,} rows: {reason}")
    return df[~mask].reset_index(drop=True)

print("Setup complete.")
print(f"Project: {PROJECT}")
print(f"Data dir: {DATA_DIR}")

Setup complete.
Project: c:\Users\Claus Hack\Documents\AAA_project_team_5
Data dir: c:\Users\Claus Hack\Documents\AAA_project_team_5\data


## 1) LOAD
Load the dataset

In [2]:
data = pl.read_csv(CSV_PATH, low_memory=False)
original_rows = data.shape[0]
print(f"Loaded {data.shape[0]:,} rows, {data.shape[1]} columns")
print(f'DATA_DIR: {data.dtypes}')

print('Example columns:')
display(data.head(2))
data = data.lazy()

Loaded 15,406,960 rows, 23 columns
DATA_DIR: [String, String, String, String, Float64, String, Int64, Int64, Int64, Int64, String, String, String, String, String, String, String, String, String, String, String, String, String]
Example columns:


Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,Fare,Tips,Tolls,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
str,str,str,str,f64,str,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str
"""f48c3b6c5c3bb929f585a167a9fd1f…","""85c39e068db414d181c6252a89fd82…","""05/01/2026 12:00:00 AM""","""05/01/2026 12:00:00 AM""",398.0,"""1,16""",null,null,32,8,"""$6,75""","""$2,00""","""$0,00""","""$1,00""","""$10,25""","""Credit Card""","""Blue Ribbon Taxi Association""","""41,878865584""","""-87,625192142""","""POINT (-87.6251921424 41.87886…","""41,899602111""","""-87,633308037""","""POINT (-87.6333080367 41.89960…"
"""ec94d6d233061d415f0c89cb9222e9…","""b3ee94a13b61037620cbbfc6a4a106…","""05/01/2026 12:00:00 AM""","""05/01/2026 12:15:00 AM""",780.0,"""2""",null,null,8,28,"""$9,50""","""$4,00""","""$0,00""","""$2,00""","""$15,50""","""Credit Card""","""Transit Administrative Center …","""41,899602111""","""-87,633308037""","""POINT (-87.6333080367 41.89960…","""41,874005383""","""-87,66351755""","""POINT (-87.6635175498 41.87400…"


## 2) TYPE CONVERSIONS + TRIP END FILLING
Next step is to convert the types to the correct data types as we can see above that not all columns are numeric.

In [3]:
# Casting
money_cols = ["Fare", "Tips", "Tolls", "Extras",  "Trip Total"]

data = data.with_columns(
    pl.col(money_cols)
    .str.replace("$", "", literal=True)
    .str.replace(",", ".", literal=True)
    .cast(pl.Float64, strict=False),
).with_columns( # Miles to float
    pl.col('Trip Miles').str.replace(",", ".", literal=True).cast(pl.Float64, strict=False)
).with_columns( # timestamp casting
    pl.col("Trip Start Timestamp").str.to_datetime("%m/%d/%Y %I:%M:%S %p").alias("Trip Start Timestamp"),
    pl.col("Trip End Timestamp").str.to_datetime("%m/%d/%Y %I:%M:%S %p").alias("Trip End Timestamp")
). with_columns([
    pl.col("Pickup Centroid Latitude").str.replace(",", ".").cast(pl.Float64),
    pl.col("Pickup Centroid Longitude").str.replace(",", ".").cast(pl.Float64),
    pl.col("Dropoff Centroid Latitude").str.replace(",", ".").cast(pl.Float64),
    pl.col("Dropoff Centroid Longitude").str.replace(",", ".").cast(pl.Float64)]
)


In [4]:
data = data.collect()

In [5]:
data.head(2)

Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,Fare,Tips,Tolls,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
str,str,datetime[μs],datetime[μs],f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,str
"""f48c3b6c5c3bb929f585a167a9fd1f…","""85c39e068db414d181c6252a89fd82…",2026-05-01 00:00:00,2026-05-01 00:00:00,398.0,1.16,null,null,32,8,6.75,2.0,0.0,1.0,10.25,"""Credit Card""","""Blue Ribbon Taxi Association""",41.878866,-87.625192,"""POINT (-87.6251921424 41.87886…",41.899602,-87.633308,"""POINT (-87.6333080367 41.89960…"
"""ec94d6d233061d415f0c89cb9222e9…","""b3ee94a13b61037620cbbfc6a4a106…",2026-05-01 00:00:00,2026-05-01 00:15:00,780.0,2.0,null,null,8,28,9.5,4.0,0.0,2.0,15.5,"""Credit Card""","""Transit Administrative Center …",41.899602,-87.633308,"""POINT (-87.6333080367 41.89960…",41.874005,-87.663518,"""POINT (-87.6635175498 41.87400…"


In [6]:
# Ensure all time columns are set
fill_condition = (
    pl.col("Trip End Timestamp").is_null()
    & pl.col("Trip Start Timestamp").is_not_null()
    & pl.col("Trip Seconds").is_not_null()
)

affected_rows = data.select(
    count=fill_condition.sum()
).item()

print(f"Number of missing end timestamps filled: {affected_rows}")

data = data.with_columns(
    pl.when(fill_condition)
    .then(pl.col("Trip Start Timestamp") + pl.duration(seconds="Trip Seconds"))
    .otherwise(pl.col("Trip End Timestamp"))
    .alias("Trip End Timestamp")
)

Number of missing end timestamps filled: 0


---
## 3) RENAME TO SNAKE_CASE

In [7]:
snake_map = {
    "Trip ID": "trip_id",
    "Taxi ID": "taxi_id",
    "Trip Start Timestamp": "trip_start",
    "Trip End Timestamp": "trip_end",
    "Trip Seconds": "trip_seconds",
    "Trip Miles": "trip_miles",
    "Pickup Census Tract": "pickup_census_tract",
    "Dropoff Census Tract": "dropoff_census_tract",
    "Pickup Community Area": "pickup_community_area",
    "Dropoff Community Area": "dropoff_community_area",
    "Fare": "fare_usd",
    "Tips": "tips_usd",
    "Tolls": "tolls_usd",
    "Extras": "extras_usd",
    "Trip Total": "trip_total_usd",
    "Payment Type": "payment_type",
    "Company": "company",
    "Pickup Centroid Latitude": "pickup_lat",
    "Pickup Centroid Longitude": "pickup_lon",
    "Pickup Centroid Location": "pickup_location",
    "Dropoff Centroid Latitude": "dropoff_lat",
    "Dropoff Centroid Longitude": "dropoff_lon",
    "Dropoff Centroid  Location": "dropoff_location",
}
# Polars: returns a new DataFrame, so reassign
data = data.rename(snake_map)
print(f"Renamed {len(snake_map)} columns.")
print(data.columns)

Renamed 23 columns.
['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare_usd', 'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type', 'company', 'pickup_lat', 'pickup_lon', 'pickup_location', 'dropoff_lat', 'dropoff_lon', 'dropoff_location']


---
## 4) LAT/LON FILL VIA CENSUS TRACT
Use known census tract centroids (median lat/lon) to fill missing lat/lon.

In [8]:
# 1. Clean the census tract columns (cast to Int64 if they aren't already)
data = data.with_columns(
    pl.col("pickup_census_tract").cast(pl.Int64, strict=False),
    pl.col("dropoff_census_tract").cast(pl.Int64, strict=False)
)

# 2. Calculate median of cencus tract lat/lon to fill the gaps
# pickup
pu_tract_map = (
    data.filter(pl.col("pickup_census_tract").is_not_null() & pl.col("pickup_lat").is_not_null())
    .group_by("pickup_census_tract")
    .agg(
        pl.col("pickup_lat").median().alias("mapped_pu_lat"),
        pl.col("pickup_lon").median().alias("mapped_pu_lon")
    )
)

#dropoff
do_tract_map = (
    data.filter(pl.col("dropoff_census_tract").is_not_null() & pl.col("dropoff_lat").is_not_null())
    .group_by("dropoff_census_tract")
    .agg(
        pl.col("dropoff_lat").median().alias("mapped_do_lat"),
        pl.col("dropoff_lon").median().alias("mapped_do_lon")
    )
)

# 3. Calculate how many rows will be affected before updating
mask_pu = pl.col("pickup_lat").is_null() & pl.col("pickup_census_tract").is_not_null()
mask_do = pl.col("dropoff_lat").is_null() & pl.col("dropoff_census_tract").is_not_null()

pu_affected = data.select(count=mask_pu.sum()).item()
do_affected = data.select(count=mask_do.sum()).item()

# 4. Join the maps and fill in the missing lat/lon values
data = (
    data
    .join(pu_tract_map, on="pickup_census_tract", how="left")
    .join(do_tract_map, on="dropoff_census_tract", how="left")
    .with_columns(
        pl.when(mask_pu).then(pl.col("mapped_pu_lat")).otherwise(pl.col("pickup_lat")).alias("pickup_lat"),
        pl.when(mask_pu).then(pl.col("mapped_pu_lon")).otherwise(pl.col("pickup_lon")).alias("pickup_lon"),
        pl.when(mask_do).then(pl.col("mapped_do_lat")).otherwise(pl.col("dropoff_lat")).alias("dropoff_lat"),
        pl.when(mask_do).then(pl.col("mapped_do_lon")).otherwise(pl.col("dropoff_lon")).alias("dropoff_lon"),
    )
    # Drop the temporary mapping columns used for the join
    .drop(["mapped_pu_lat", "mapped_pu_lon", "mapped_do_lat", "mapped_do_lon"])
)

# 5. Print out the audit counts
print(f"Filled {pu_affected:,} pickup lat/lon from census tract")
print(f"Filled {do_affected:,} dropoff lat/lon from census tract")

Filled 2,023 pickup lat/lon from census tract
Filled 16,403 dropoff lat/lon from census tract


In [9]:
data.head(3)

trip_id,taxi_id,trip_start,trip_end,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,fare_usd,tips_usd,tolls_usd,extras_usd,trip_total_usd,payment_type,company,pickup_lat,pickup_lon,pickup_location,dropoff_lat,dropoff_lon,dropoff_location
str,str,datetime[μs],datetime[μs],f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,str
"""f48c3b6c5c3bb929f585a167a9fd1f…","""85c39e068db414d181c6252a89fd82…",2026-05-01 00:00:00,2026-05-01 00:00:00,398.0,1.16,null,null,32,8,6.75,2.0,0.0,1.0,10.25,"""Credit Card""","""Blue Ribbon Taxi Association""",41.878866,-87.625192,"""POINT (-87.6251921424 41.87886…",41.899602,-87.633308,"""POINT (-87.6333080367 41.89960…"
"""ec94d6d233061d415f0c89cb9222e9…","""b3ee94a13b61037620cbbfc6a4a106…",2026-05-01 00:00:00,2026-05-01 00:15:00,780.0,2.0,null,null,8,28,9.5,4.0,0.0,2.0,15.5,"""Credit Card""","""Transit Administrative Center …",41.899602,-87.633308,"""POINT (-87.6333080367 41.89960…",41.874005,-87.663518,"""POINT (-87.6635175498 41.87400…"
"""e96d8e910cc577d191447dfdf5dc59…","""0c758d798a3980ff73e5f1a334db48…",2026-05-01 00:00:00,2026-05-01 00:00:00,36.0,0.18,null,null,56,56,3.5,0.0,0.0,0.0,3.5,"""Cash""","""Sun Taxi""",41.792592,-87.769615,"""POINT (-87.7696154528 41.79259…",41.792592,-87.769615,"""POINT (-87.7696154528 41.79259…"


In [10]:
df = data.to_pandas()

---
## 5) CENSUS TRACT IMPUTATION VIA LOOKUP (lat/lon -> census tract)
Since lat/lon are centroid coordinates of census tracts, identical (lat, lon) pairs
always map to the same census tract. We build a simple lookup dictionary from rows
where both lat/lon and census tract are known, then map missing tracts.

In [11]:
for prefix in ["pickup", "dropoff"]:
    lat_col = f"{prefix}_lat"
    lon_col = f"{prefix}_lon"
    tract_col = f"{prefix}_census_tract"

    known = df[df[tract_col].notna() & df[lat_col].notna() & df[lon_col].notna()]
    lookup = known.groupby([lat_col, lon_col])[tract_col].first().to_dict()
    print(f"{prefix}: Built lookup with {len(lookup):,} unique (lat,lon) -> census tract pairs")

    missing = df[df[tract_col].isna() & df[lat_col].notna() & df[lon_col].notna()]
    if len(missing) == 0:
        print(f"{prefix}: No missing census tracts to impute.")
        continue

    keys = list(zip(missing[lat_col], missing[lon_col]))
    mapped = pd.Series([lookup.get(k, np.nan) for k in keys], index=missing.index)
    filled = mapped.notna().sum()
    df.loc[missing.index, tract_col] = mapped

    print(f"{prefix}: Imputed {filled:,} / {len(missing):,} missing census tracts via lookup")

pickup: Built lookup with 610 unique (lat,lon) -> census tract pairs
pickup: Imputed 5,381 / 8,128,006 missing census tracts via lookup
dropoff: Built lookup with 657 unique (lat,lon) -> census tract pairs
dropoff: Imputed 5,928 / 7,482,644 missing census tracts via lookup


---
## 6) DROP ROWS WITH NO SPATIAL DATA
Remove rows where ALL spatial columns are null. Save to separate parquet.

In [12]:
spatial_cols = [
    "pickup_census_tract", "dropoff_census_tract",
    "pickup_community_area", "dropoff_community_area",
    "pickup_lat", "pickup_lon", "pickup_location",
    "dropoff_lat", "dropoff_lon", "dropoff_location",
]

no_spatial_mask = df[spatial_cols].isna().all(axis=1)
no_spatial_count = no_spatial_mask.sum()
df[no_spatial_mask].to_parquet(str(OUTPUT_NO_SPATIAL), index=False)
df = df[~no_spatial_mask].reset_index(drop=True)

print(f"Removed {no_spatial_count:,} rows with no spatial data at all")
print(f"Remaining: {len(df):,} rows")

Removed 279,212 rows with no spatial data at all
Remaining: 15,127,748 rows


---
## 7) PAYMENT TYPE CLEANUP

In [13]:
df["payment_type"] = (
    df["payment_type"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
)
print(f"Payment types: {df['payment_type'].nunique()}")
print(df["payment_type"].value_counts())

Payment types: 8
payment_type
credit card    5553409
cash           3794159
mobile         3554075
prcard         1650607
unknown         526895
no charge        40623
dispute           7916
prepaid             64
Name: count, dtype: int64


---
## 8) COMPANY STANDARDIZATION

In [14]:
df["company"] = (
    df["company"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
)
df["company"] = df["company"].str.replace(r"[,\.]", "", regex=True)
df["company"] = df["company"].str.strip().str.replace(r"\s+", " ", regex=True)

manual_mappings = {
    "choice taxi association inc": "choice taxi association",
    "blue ribbon taxi association inc": "blue ribbon taxi association",
    "top cab affiliation": "top cab",
    "medallion leasin": "medallion leasing",
}
df["company"] = df["company"].replace(manual_mappings)

company_export = df["company"].value_counts().reset_index()
company_export.columns = ["company_name", "count"]
company_export.to_csv(str(OUTPUT_COMPANIES), index=False)
print(f"Unique companies: {df['company'].nunique()}")

Unique companies: 39


---
## 9) DUPLICATE CHECK (trip_id)

In [15]:
total = len(df)
unique = df["trip_id"].nunique()
dups = total - unique
print(f"Total rows: {total:,}, Unique trip_ids: {unique:,}, Duplicates: {dups:,}")

if dups > 0:
    dup_mask = df.duplicated(subset="trip_id", keep="first")
    df = flag_and_remove(df, dup_mask, "duplicate_trip_id")
else:
    print("No duplicates found.")

Total rows: 15,127,748, Unique trip_ids: 15,127,748, Duplicates: 0
No duplicates found.


---
## 10) REMOVE INVALID TRIPS
Remove if **any** of:
- `trip_seconds < 60` (under 1 minute)
- `trip_miles < 0.01` (under 0.01 miles)
- `fare_usd == 0` (zero fare)

All removed rows go into `Removed_Rows.parquet` with a reason.

In [16]:
neg_cols = ["trip_seconds", "trip_miles", "fare_usd", "tips_usd", "tolls_usd", "extras_usd", "trip_total_usd"]
for col in neg_cols:
    df = flag_and_remove(df, df[col] < 0, f"negative_{col}")

df = flag_and_remove(
    df,
    (df["trip_seconds"].notna()) & (df["trip_seconds"] < 60),
    "trip_duration_under_1min",
)

df = flag_and_remove(
    df,
    (df["trip_miles"].notna()) & (df["trip_miles"] < 0.01),
    "trip_distance_under_0.01mi",
)

df = flag_and_remove(
    df,
    (df["fare_usd"].notna()) & (df["fare_usd"] == 0),
    "fare_zero",
)

print(f"\nRemaining after invalid trip removal: {len(df):,}")

  Removed 7,685,364 rows: trip_duration_under_1min
  Removed 384,315 rows: trip_distance_under_0.01mi
  Removed 2,920 rows: fare_zero

Remaining after invalid trip removal: 7,055,149


---
## 11) CONSISTENCY & LOGIC CHECKS

In [17]:
df = flag_and_remove(
    df,
    df["trip_end"] < df["trip_start"],
    "time_travel_end_before_start",
)

ts_safe = df["trip_seconds"].fillna(0).to_numpy(dtype="float64")
tm_safe = df["trip_miles"].fillna(0).to_numpy(dtype="float64")
speed_mph = np.where(ts_safe > 0, tm_safe / (ts_safe / 3600), 0)
df = flag_and_remove(
    df,
    speed_mph > 100,
    "impossible_speed_over_100mph",
)

print(f"Remaining after consistency checks: {len(df):,}")

  Removed 76 rows: time_travel_end_before_start


C:\Users\Claus Hack\AppData\Local\Temp\ipykernel_27156\4063605070.py:9: RuntimeWarning: divide by zero encountered in divide
  speed_mph = np.where(ts_safe > 0, tm_safe / (ts_safe / 3600), 0)
C:\Users\Claus Hack\AppData\Local\Temp\ipykernel_27156\4063605070.py:9: RuntimeWarning: invalid value encountered in divide
  speed_mph = np.where(ts_safe > 0, tm_safe / (ts_safe / 3600), 0)


  Removed 2,914 rows: impossible_speed_over_100mph
Remaining after consistency checks: 7,052,159


---
## 12) ABSOLUTE OUTLIER FILTERING
Hard cutoffs:
- `trip_miles > 200` -> remove
- `trip_seconds > 7200` (> 2h) -> remove
- `fare_usd < 3.25` (below basefare) -> remove
- No upper cut on fare

In [18]:
df = flag_and_remove(
    df,
    (df["trip_miles"].notna()) & (df["trip_miles"] > 200),
    "trip_miles_over_200",
)

df = flag_and_remove(
    df,
    (df["trip_seconds"].notna()) & (df["trip_seconds"] > 7200),
    "trip_seconds_over_2h",
)

df = flag_and_remove(
    df,
    (df["fare_usd"].notna()) & (df["fare_usd"] < 3.25),
    "fare_below_basefare_3.25",
)

print(f"\nRemaining after absolute outlier filtering: {len(df):,}")

  Removed 38 rows: trip_miles_over_200
  Removed 618 rows: fare_below_basefare_3.25

Remaining after absolute outlier filtering: 7,051,503


---
## 13) DROP LOCATION TEXT COLUMNS & SAVE CLEANED PARQUET
Keep census tract columns (needed later). Drop only the point-location text columns.

In [19]:
cols_to_drop = ["pickup_location", "dropoff_location"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Final columns: {df.columns.tolist()}")
print(f"Final rows: {len(df):,}")

df.to_parquet(str(OUTPUT_CLEAN), index=False)
print(f"Saved {OUTPUT_CLEAN}")

Final columns: ['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare_usd', 'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type', 'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon']
Final rows: 7,051,503
Saved c:\Users\Claus Hack\Documents\AAA_project_team_5\data\Taxi_Trips_cleaned.parquet


---
## 14) ID REMAPPING + COMPACT PARQUET

In [20]:
trip_id_map = pd.DataFrame({
    "trip_id_int": range(1, len(df) + 1),
    "trip_id_orig": df["trip_id"].values,
})

taxi_ids_unique = df["taxi_id"].dropna().unique()
taxi_id_map = pd.DataFrame({
    "taxi_id_int": range(1, len(taxi_ids_unique) + 1),
    "taxi_id_orig": taxi_ids_unique,
})
taxi_lookup = dict(zip(taxi_id_map["taxi_id_orig"], taxi_id_map["taxi_id_int"]))

df["trip_id_int"] = range(1, len(df) + 1)
df["taxi_id_int"] = df["taxi_id"].map(taxi_lookup)

df.drop(columns=["trip_id", "taxi_id"], inplace=True)

cols_order = ["trip_id_int", "taxi_id_int"] + [c for c in df.columns if c not in ("trip_id_int", "taxi_id_int")]
df = df[cols_order]

trip_id_map.to_csv(str(OUTPUT_TRIP_ID_MAP), index=False)
taxi_id_map.to_csv(str(OUTPUT_TAXI_ID_MAP), index=False)

print(f"trip_id_map: {len(trip_id_map):,} rows")
print(f"taxi_id_map: {len(taxi_id_map):,} rows")

trip_id_map: 7,051,503 rows
taxi_id_map: 3,308 rows


---
## 15) TIME FEATURES FROM trip_start

In [21]:
ts = df["trip_start"]

df["hour"] = ts.dt.hour.astype("int8")
df["day_of_week"] = ts.dt.dayofweek.astype("int8")
df["month"] = ts.dt.month.astype("int8")
df["date"] = ts.dt.date
df["is_weekend"] = df["day_of_week"].isin([5, 6])

df["bin_30min"] = ts.dt.floor("30min")
df["bin_1h"] = ts.dt.floor("1h")
df["bin_4h"] = ts.dt.floor("4h")
df["bin_1d"] = ts.dt.normalize()
df["bin_1w"] = ts.dt.to_period("W").apply(lambda p: p.start_time)

time_cols = ["hour", "day_of_week", "month", "date", "is_weekend",
            "bin_30min", "bin_1h", "bin_4h", "bin_1d", "bin_1w"]

print(f"Added {len(time_cols)} time columns.")
df[time_cols].sample(10, random_state=42)

Added 10 time columns.


,hour,day_of_week,month,date,is_weekend,bin_30min,bin_1h,bin_4h,bin_1d,bin_1w
1820248,8,6,10,2025-10-19,True,2025-10-19 08:00:00,2025-10-19 08:00:00,2025-10-19 08:00:00,2025-10-19,2025-10-13
6087598,22,5,5,2024-05-18,True,2024-05-18 22:30:00,2024-05-18 22:00:00,2024-05-18 20:00:00,2024-05-18,2024-05-13
6744312,15,1,2,2024-02-20,False,2024-02-20 15:00:00,2024-02-20 15:00:00,2024-02-20 12:00:00,2024-02-20,2024-02-19
3976096,12,6,2,2025-02-23,True,2025-02-23 12:30:00,2025-02-23 12:00:00,2025-02-23 12:00:00,2025-02-23,2025-02-17
6448588,12,0,4,2024-04-01,False,2024-04-01 12:30:00,2024-04-01 12:00:00,2024-04-01 12:00:00,2024-04-01,2024-04-01
3707192,8,3,3,2025-03-27,False,2025-03-27 08:30:00,2025-03-27 08:00:00,2025-03-27 08:00:00,2025-03-27,2025-03-24
4446411,15,0,12,2024-12-16,False,2024-12-16 15:30:00,2024-12-16 15:00:00,2024-12-16 12:00:00,2024-12-16,2024-12-16
1173867,15,3,12,2025-12-25,False,2025-12-25 15:30:00,2025-12-25 15:00:00,2025-12-25 12:00:00,2025-12-25,2025-12-22
2375155,12,2,8,2025-08-20,False,2025-08-20 12:30:00,2025-08-20 12:00:00,2025-08-20 12:00:00,2025-08-20,2025-08-18
4440504,11,1,12,2024-12-17,False,2024-12-17 11:30:00,2024-12-17 11:00:00,2024-12-17 08:00:00,2024-12-17,2024-12-16


---
## 16) CYCLIC ENCODING (sin/cos)

In [22]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

cyclic_cols = ["hour", "hour_sin", "hour_cos",
              "day_of_week", "dow_sin", "dow_cos",
              "month", "month_sin", "month_cos"]

print(f"Added 6 cyclic encoding columns.")
df[cyclic_cols].sample(10, random_state=42)

Added 6 cyclic encoding columns.


,hour,hour_sin,hour_cos,day_of_week,dow_sin,dow_cos,month,month_sin,month_cos
1820248,8,8.660254e-01,-0.500000,6,-0.781831,0.623490,10,-8.660254e-01,5.000000e-01
6087598,22,-5.000000e-01,0.866025,5,-0.974928,-0.222521,5,5.000000e-01,-8.660254e-01
6744312,15,-7.071068e-01,-0.707107,1,0.781831,0.623490,2,8.660254e-01,5.000000e-01
3976096,12,1.224647e-16,-1.000000,6,-0.781831,0.623490,2,8.660254e-01,5.000000e-01
6448588,12,1.224647e-16,-1.000000,0,0.000000,1.000000,4,8.660254e-01,-5.000000e-01
3707192,8,8.660254e-01,-0.500000,3,0.433884,-0.900969,3,1.000000e+00,6.123234e-17
4446411,15,-7.071068e-01,-0.707107,0,0.000000,1.000000,12,-2.449294e-16,1.000000e+00
1173867,15,-7.071068e-01,-0.707107,3,0.433884,-0.900969,12,-2.449294e-16,1.000000e+00
2375155,12,1.224647e-16,-1.000000,2,0.974928,-0.222521,8,-8.660254e-01,-5.000000e-01
4440504,11,2.588190e-01,-0.965926,1,0.781831,0.623490,12,-2.449294e-16,1.000000e+00


---
## 17) ELECTRONIC PAYMENT FEE DETECTION + SAVE COMPACT PARQUET

In [23]:
df["has_electronic_fee"] = (
    df["payment_type"].isin(["credit card", "mobile"])
    & ((df["trip_total_usd"] - df[["fare_usd","tips_usd","tolls_usd","extras_usd"]].sum(axis=1)).round(2) > 0.49)
    & ((df["trip_total_usd"] - df[["fare_usd","tips_usd","tolls_usd","extras_usd"]].sum(axis=1)).round(2) < 0.51)
)

df.to_parquet(str(OUTPUT_COMPACT), index=False)
print(f"Saved compact parquet: {OUTPUT_COMPACT}")
print(f"has_electronic_fee: {df['has_electronic_fee'].sum():,} rows")

Saved compact parquet: c:\Users\Claus Hack\Documents\AAA_project_team_5\data\Taxi_Trips_compact.parquet
has_electronic_fee: 3,408,440 rows


---
## 18) SAVE REMOVED ROWS

In [24]:
print(original_rows)

15406960


In [25]:
if removed_frames:
    removed_df = pd.concat(removed_frames, ignore_index=True)
    removed_df.to_parquet(str(OUTPUT_REMOVED), index=False)
    print(f"Saved {len(removed_df):,} removed rows to {OUTPUT_REMOVED}")
    print(f"\nRemoval breakdown:")
    print(removed_df["removal_reason"].value_counts().to_string())
else:
    print("No rows were removed.")

print(f"\n=== CLEANING SUMMARY ===")
print(f"Original rows:  {original_rows:,}")
print(f"Final rows:     {len(df):,}")
removed_total = original_rows - len(df)
print(f"Removed:        {removed_total:,} ({removed_total/original_rows*100:.2f}%)")

Saved 8,076,245 removed rows to c:\Users\Claus Hack\Documents\AAA_project_team_5\data\Removed_Rows.parquet

Removal breakdown:
removal_reason
trip_duration_under_1min        7685364
trip_distance_under_0.01mi       384315
fare_zero                          2920
impossible_speed_over_100mph       2914
fare_below_basefare_3.25            618
time_travel_end_before_start         76
trip_miles_over_200                  38

=== CLEANING SUMMARY ===
Original rows:  15,406,960
Final rows:     7,051,503
Removed:        8,355,457 (54.23%)
